# Build email extractor

In [ ]:
from pydantic import BaseModel, Field, EmailStr 
from typing import Optional, Literal 
from pydantic_ai import Agent

class EmailExtractor(BaseModel):
    sender_name: Optional[str]
    sender_email: EmailStr
    issue_category: Literal["billing", "damaged product", "technical", "other"]
    urgency: Literal["low", "medium", "high"]
    summary: str = Field(description="summarize the email in 3-4 sentences")


email_extractor_agent = Agent(model="openrouter:nvidia/nemotron-3-super-120b-a12b:free", 
system_prompt="""You are a costumer support agent, your task is to extract relevant information from an email"""
, output_type=EmailExtractor, retries=1)




In [15]:
import pandas as pd
df = pd.read_json("customer_service_emails.json")
df

,input,expectations
0,{'email': 'From: Erik Lindqvist <erik.lindqvis...,"{'sender_name': 'Erik Lindqvist', 'sender_emai..."
1,{'email': 'From: Maja Bergström <maja.bergstro...,"{'sender_name': 'Maja Bergström', 'sender_emai..."
2,{'email': 'From: Oscar Johansson <oscar.johans...,"{'sender_name': 'Oscar Johansson', 'sender_ema..."
3,{'email': 'From: Linnea Karlsson <linnea.karls...,"{'sender_name': 'Linnea Karlsson', 'sender_ema..."


In [16]:
sample_mail = df.iloc[2]["input"]["email"]
sample_mail

"From: Oscar Johansson <oscar.johansson@yahoo.se>\nSubject: Cannot access my account for 3 days - Urgent help needed\n\nHello Support Team,\n\nI am reaching out because I have been completely locked out of my account for the past three days and I am running out of ideas on how to fix this on my own. The problem started on Monday evening when I tried to log in as usual but kept receiving an 'Invalid credentials' error despite being absolutely certain that I was entering the correct password.\n\nI followed the instructions on your website to reset my password, but the problem is that the password reset email never arrives in my inbox. I have checked my spam and junk folders multiple times, and there is nothing there either. I have attempted the reset process at least six or seven times across different browsers and even from my phone, but the result is always the same - no email arrives.\n\nThis is causing me real problems because I have important documents and data stored in my account 

In [17]:
result = await email_extractor_agent.run(sample_mail)
print(result)

AgentRunResult(output=EmailExtractor(sender_name='Oscar Johansson', sender_email='oscar.johansson@yahoo.se', issue_category='technical', urgency='high', summary="Oscar Johansson has been locked out of his account for three days due to 'Invalid credentials' errors despite entering the correct password. He has attempted multiple password reset procedures across different browsers and devices, but reset emails are not arriving in his inbox (including spam/junk folders). He urgently needs access to important work documents to meet an upcoming Friday deadline and requests immediate assistance to verify his identity and regain account access."))


In [18]:
result.output

EmailExtractor(sender_name='Oscar Johansson', sender_email='oscar.johansson@yahoo.se', issue_category='technical', urgency='high', summary="Oscar Johansson has been locked out of his account for three days due to 'Invalid credentials' errors despite entering the correct password. He has attempted multiple password reset procedures across different browsers and devices, but reset emails are not arriving in his inbox (including spam/junk folders). He urgently needs access to important work documents to meet an upcoming Friday deadline and requests immediate assistance to verify his identity and regain account access.")

## Load in prompts from mlflow

In [19]:
from mlflow.genai import load_prompt

class EmailExtractor(BaseModel):
    sender_name: Optional[str]
    sender_email: EmailStr
    issue_category: Literal["billing", "damaged product", "technical", "other"]
    urgency: Literal["low", "medium", "high"] = Field(
        description=load_prompt("email-urgency-description").format()
    )
    summary: str = Field(
        description=load_prompt("summary-description").format(num_sentences=4)
    )


email_extractor_agent = Agent(
    model="openrouter:nvidia/nemotron-3-super-120b-a12b:free",
    system_prompt=load_prompt("email-extractor-system-prompt").format(),
    output_type=EmailExtractor,
)

In [20]:
result = await email_extractor_agent.run(sample_mail)
result.output

EmailExtractor(sender_name='Oscar Johansson', sender_email='oscar.johansson@yahoo.se', issue_category='technical', urgency='high', summary='Oscar Johansson has been locked out of his account for three days due to invalid credentials. He has attempted password reset multiple times but reset emails are not arriving in his inbox or spam folder. This is preventing him from accessing important work documents with a Friday deadline. He requests urgent help to regain access and is willing to verify his identity.')

## LLM judge
- required data with columns: inputs, expectations, outputs
- mlflow experiments
- 

In [21]:
result.output.model_dump()

{'sender_name': 'Oscar Johansson',
 'sender_email': 'oscar.johansson@yahoo.se',
 'issue_category': 'technical',
 'urgency': 'high',
 'summary': 'Oscar Johansson has been locked out of his account for three days due to invalid credentials. He has attempted password reset multiple times but reset emails are not arriving in his inbox or spam folder. This is preventing him from accessing important work documents with a Friday deadline. He requests urgent help to regain access and is willing to verify his identity.'}

In [22]:
df

,input,expectations
0,{'email': 'From: Erik Lindqvist <erik.lindqvis...,"{'sender_name': 'Erik Lindqvist', 'sender_emai..."
1,{'email': 'From: Maja Bergström <maja.bergstro...,"{'sender_name': 'Maja Bergström', 'sender_emai..."
2,{'email': 'From: Oscar Johansson <oscar.johans...,"{'sender_name': 'Oscar Johansson', 'sender_ema..."
3,{'email': 'From: Linnea Karlsson <linnea.karls...,"{'sender_name': 'Linnea Karlsson', 'sender_ema..."


## Todo: dataframe with inputs, expectations, outputs but only 1

In [25]:
df["outputs"] = [{}, {},result.output.model_dump(), {}]
df

,input,expectations,outputs
0,{'email': 'From: Erik Lindqvist <erik.lindqvis...,"{'sender_name': 'Erik Lindqvist', 'sender_emai...",{}
1,{'email': 'From: Maja Bergström <maja.bergstro...,"{'sender_name': 'Maja Bergström', 'sender_emai...",{}
2,{'email': 'From: Oscar Johansson <oscar.johans...,"{'sender_name': 'Oscar Johansson', 'sender_ema...","{'sender_name': 'Oscar Johansson', 'sender_ema..."
3,{'email': 'From: Linnea Karlsson <linnea.karls...,"{'sender_name': 'Linnea Karlsson', 'sender_ema...",{}


In [32]:
df.sample = df.drop([0,1,3])
df

,input,expectations,outputs
0,{'email': 'From: Erik Lindqvist <erik.lindqvis...,"{'sender_name': 'Erik Lindqvist', 'sender_emai...",{}
1,{'email': 'From: Maja Bergström <maja.bergstro...,"{'sender_name': 'Maja Bergström', 'sender_emai...",{}
2,{'email': 'From: Oscar Johansson <oscar.johans...,"{'sender_name': 'Oscar Johansson', 'sender_ema...","{'sender_name': 'Oscar Johansson', 'sender_ema..."
3,{'email': 'From: Linnea Karlsson <linnea.karls...,"{'sender_name': 'Linnea Karlsson', 'sender_ema...",{}


## LLM Judge

In [35]:
from mlflow.genai.scorers import get_all_scorers

# get_all_scorers()

In [ ]:
from mlflow.genai.scorers import Correctness, Summarization, Completeness, Fluency
import mlflow
llm_judge = "openrouter:nvidia/nemotron-3-nano-30b-a3b:free"

with mlflow.start_run(run_name="email_extractor_evaluation"):
    mlflow.log_param("model", llm_judge)

results = mlflow.genai.evaluate(
    data = df_sample,
    scorers=[
        Correctness(model=llm_judge),
        Summarization(model=llm_judge)
    ]
)

results 

NameError: name 'df_sample' is not defined

: 